In [22]:
import ollama
import os
import re
import pandas as pd
import spacy
from ipywidgets import FloatProgress
from tqdm import tqdm

In [23]:
# Load English tokenizer, tagger, parser and NER
nlp = spacy.load("en_core_web_sm")

In [24]:
# OLLAMA_BASE_URL in .env is the OpenAI-compat endpoint (ends in /v1); the ollama client needs the native API root.
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").removesuffix("/v1")
ollama_client = ollama.Client(host=OLLAMA_BASE_URL)
# gpt-oss:20b is a reasoning model that spends its token budget on hidden "thinking" before answering,
# leaving nothing for the actual response within a short num_predict. llama3.2 has no thinking mode,
# so it answers directly and fits this short classification task.
OLLAMA_MODEL = "llama3.2:latest"

In [19]:
movies = pd.read_csv("netflix_titles.csv")
movies = movies.sample(n=1000)
movies.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
4993,s4994,Movie,The Outsider,Martin Zandvliet,"Jared Leto, Tadanobu Asano, Kippei Shiina, Shi...",United States,"March 9, 2018",2018,TV-MA,121 min,"Dramas, Thrillers",An American soldier imprisoned in postwar Japa...
7822,s7823,Movie,Raiders!: The Story of the Greatest Fan Film E...,"Jeremy Coon, Tim Skousen",NaN,United States,"September 16, 2016",2015,NR,93 min,Documentaries,Three childhood friends attempt to finish the ...


In [25]:
def predict_genres(movie_description):
    prompt = f"Predict the top three genres (only genres, not descriptions) for a movie with the following description: {movie_description}"
    response = ollama_client.generate(
        model=OLLAMA_MODEL,
        prompt=prompt,
        options={"num_predict": 50, "temperature": 0.2},
    )
    predicted_genres = response.response.strip()
    return predicted_genres

In [26]:
# Create an empty list to store the predicted genres
all_predicted_genres = []
# Create an empty set to store unique genres
unique_genres_set = set()
# Iterate through the movie descriptions
for index, row in tqdm(movies.iterrows(), total=movies.shape[0]):    
        # Get the movie description
        movie_description = row['description']    
        # Predict the genres for the movie description
        predicted_genres = predict_genres(movie_description)    
        # Extract genres from the text
        predicted_genres_tokens = nlp(predicted_genres)
        predicted_genres_tokens = predicted_genres_tokens.text        
        # Use regular expression to extract genres
        genres_with_numbers = re.findall(r'\d+\.\s*([^\n]+)', predicted_genres_tokens)        
        # Remove leading/trailing whitespaces from each genre
        predicted_genres = [genre.strip().lower() for genre in genres_with_numbers]    
        # Update the set of unique genres
        unique_genres_set.update(predicted_genres)
# Convert the set of unique genres back to a list
all_unique_genres = list(unique_genres_set)

100%|██████████| 1000/1000 [07:03<00:00,  2.36it/s]


In [27]:
all_unique_genres = pd.DataFrame(all_unique_genres,columns=['genre'])
all_unique_genres.head()

,genre
0,women's fiction
1,coming-of-age
2,psychological thriller
3,heartwarming
4,spiritual


In [28]:
all_unique_genres.to_csv("genres_taxonomy_quick.csv")